In [32]:
import torch
import torch.nn as nn

import numpy as np

import mne
import matplotlib.pyplot as plt

mne.set_log_level("WARNING") # to suppress info messages

In [12]:
num_features = 40
kernel_size_temporal = 30
kernel_size_spatial = 14
kernel_size_avg_pool = 15
ffn_embedding_dim = 80
vocab_size = 3

temporal_conv = nn.Conv2d(in_channels=1,
                        out_channels=num_features,
                        kernel_size=(1, kernel_size_temporal),
                        padding="same")

spatial_conv = nn.Conv2d(in_channels=num_features,
                                       out_channels=num_features,
                                       kernel_size=(kernel_size_spatial, 1),
                                       padding="valid")

batch_norm1 = nn.BatchNorm2d(num_features)
batch_norm2 = nn.BatchNorm2d(num_features)

elu = nn.ELU()

avg_pool = nn.AvgPool2d(kernel_size=(1, kernel_size_avg_pool))

fc1 = nn.Linear(2400, ffn_embedding_dim)
fc2 = nn.Linear(ffn_embedding_dim, vocab_size)

softmax = nn.Softmax(dim=1)

In [13]:
x = torch.randn(32, 14, 900) # (N, C, T) where H is num_channels

x = x.unsqueeze(1) # (N, C, H, W) where C is 1 and H is num_channels

x = temporal_conv(x) # (32, 40, 14, 900)
x = batch_norm1(x) # (32, 40, 14, 900)
x = elu(x) # (32, 40, 14, 900)
print("before:", x.shape)

x = spatial_conv(x) # (32, 40, 1, 900)
x = batch_norm2(x) # (32, 40, 1, 900)
x = elu(x) # (32, 40, 1, 900)
print("after:", x.shape)

x = avg_pool(x) # (32, 40, 1, 60)
print("pooled:", x.shape)

x = torch.flatten(x, 1) # (32, 40 * 1 * 60) = (32, 2400)
print("flattened:", x.shape)

x = fc1(x)
x = fc2(x)
print("dense:", x.shape)

x = softmax(x) # (32, 3) - each row is a probability distribution across the 3 classes
print("softmax:", x.shape)
print(x)

before: torch.Size([32, 40, 14, 900])
after: torch.Size([32, 40, 1, 900])
pooled: torch.Size([32, 40, 1, 60])
flattened: torch.Size([32, 2400])
dense: torch.Size([32, 3])
softmax: torch.Size([32, 3])
tensor([[0.3187, 0.3313, 0.3499],
        [0.3156, 0.3240, 0.3604],
        [0.3245, 0.3716, 0.3039],
        [0.3166, 0.3629, 0.3206],
        [0.3276, 0.3649, 0.3075],
        [0.3144, 0.3783, 0.3072],
        [0.3137, 0.3540, 0.3323],
        [0.3158, 0.3366, 0.3475],
        [0.3251, 0.3451, 0.3298],
        [0.3332, 0.3453, 0.3215],
        [0.3098, 0.3384, 0.3518],
        [0.3099, 0.3690, 0.3211],
        [0.3322, 0.3270, 0.3408],
        [0.2809, 0.3595, 0.3596],
        [0.3491, 0.3654, 0.2856],
        [0.3170, 0.3603, 0.3227],
        [0.3155, 0.3386, 0.3459],
        [0.3491, 0.3470, 0.3039],
        [0.3468, 0.3516, 0.3016],
        [0.3240, 0.3546, 0.3215],
        [0.3292, 0.3620, 0.3087],
        [0.3140, 0.4026, 0.2834],
        [0.3506, 0.3201, 0.3293],
        [0.3079, 0

In [14]:
raw = mne.io.read_raw_edf("/var/log/thavamount/eeg_dataset/motor_eeg/1.0.0/S001/S001R03.edf", preload=True)

In [38]:
events, event_ids = mne.events_from_annotations(raw)
epochs = mne.Epochs(raw, events=events)

In [39]:
print(raw.get_data().shape) # (64, 20000)
print(events.shape) # every row is [sample, 0, event_id]
print(epochs["2"].get_data().shape) # corresponding to right hand, (n_epochs, C, T)
print(epochs.get_data().shape) # (29, 64, 113)

(64, 20000)
(30, 3)
(8, 64, 113)
(29, 64, 113)


In [40]:
# we want to take the differences between the first column numbers in events to get the time duration of each epoch
event_space = events[1:, 0] - events[:-1, 0]
print(event_space)

[672 656 672 656 672 656 672 656 672 656 672 656 672 656 672 656 672 656
 672 656 672 656 672 656 672 656 672 656 672]


In [58]:
events, event_ids = mne.events_from_annotations(raw)
events = np.delete(events, 1, axis=1) # every row: (sample, event_id)
print(events)
print(epochs.get_data())

[[    0     1]
 [  672     3]
 [ 1328     1]
 [ 2000     2]
 [ 2656     1]
 [ 3328     2]
 [ 3984     1]
 [ 4656     3]
 [ 5312     1]
 [ 5984     3]
 [ 6640     1]
 [ 7312     2]
 [ 7968     1]
 [ 8640     2]
 [ 9296     1]
 [ 9968     3]
 [10624     1]
 [11296     2]
 [11952     1]
 [12624     3]
 [13280     1]
 [13952     3]
 [14608     1]
 [15280     2]
 [15936     1]
 [16608     2]
 [17264     1]
 [17936     3]
 [18592     1]
 [19264     2]]
[[[-1.50000000e-05 -6.00000000e-06  4.00000000e-06 ... -5.40000000e-05
   -4.80000000e-05 -6.00000000e-05]
  [ 2.90909091e-06 -4.09090909e-06  2.90909091e-06 ... -3.70909091e-05
   -3.50909091e-05 -4.70909091e-05]
  [ 3.48484848e-06  9.48484848e-06  6.48484848e-06 ... -2.85151515e-05
   -3.15151515e-05 -4.25151515e-05]
  ...
  [ 1.71818182e-05  1.01818182e-05  1.41818182e-05 ... -1.38181818e-05
   -1.58181818e-05  1.81818182e-07]
  [ 9.03030303e-06 -2.96969697e-06 -6.96969697e-06 ... -4.09696970e-05
   -6.09696970e-05 -4.19696970e-05]
  [-4.48

In [59]:
print(len(events))
print(len(epochs.events))
print(epochs.drop_log)

30
29
(('NO_DATA',), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), ())
